# Estudio 06 · Fase 1a · Más historia (solo epidemiología)

Comparación 1a, preregistrada en `code/preregistration.py → PHASE1` (commit `6a29a8e`; enmienda G1 en `81e72a4`):

- **L:** C3 (`base_6` + B3) · E3 · S4_s3 con los hiperparámetros de la V1, entrenada desde 2005; bloques por temporada
  2008–2024 y `calendario_2025` (2008 solo calibra).
- **R:** la misma configuración entrenada desde 2017 con la tabla de orígenes exacta de la V1; bloques 2018–2024 y
  `calendario_2025` (2018 solo calibra).
- Platt de cada bloque con todas las temporadas cerradas anteriores; alerta p ≥ 0,50; 5 semillas.
- **Deciden 2022 y 2023** (R no tiene entrenamiento antes de 2017); 2024 es contraste. Se adopta L si gana o empata.

Nada se escribe en `data/`; las predicciones van a `models/experimentos/mejora_modelo/fase1a/` (no versionado).

In [ ]:
import sys

import pandas as pd

from denguekay.config.paths import RESEARCH
from denguekay.evaluation import episodes as diag
from denguekay.evaluation import protocol as pa
from denguekay.model import tracking
from denguekay.model.device import xgboost_settings

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase1_history as p1
import preregistration as pr

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
METRICS = STUDY / "docs" / "metricas" / "fase1a.json"
device = xgboost_settings()
inp = p1.phase1a_inputs()
device

## 1 · Guardas antes de mirar resultados (G1 enmendada, G2, G3; G4 es un test)

In [ ]:
g = p1.guards(inp, pr.G1_EXCEPTION)
failed = [k for k, v in g.items() if not v["pasa"]]
if failed:
    raise RuntimeError(f"Fallan las guardas {failed}: se detiene la fase. {g}")
g

## 2 · Entrenamiento y evaluación (5 semillas)

In [ ]:
L = p1.run_variant(inp["matrices_L"], inp["origenes_L"], p1.seasonal_blocks(p1.FIRST_BLOCK_SEASON_LONG),
                   inp["serie"], pr.SEEDS)
R = p1.run_variant(inp["matrices_R"], inp["origenes_R"], p1.seasonal_blocks(p1.FIRST_BLOCK_SEASON_2017),
                   inp["serie"], pr.SEEDS)
saved_v1 = p1.read_saved(p1.V1_SAVED)
V1 = {"evaluaciones": {s: pa.evaluate(g_.drop(columns="semilla"), inp["serie"])
                       for s, g_ in saved_v1.groupby("semilla")}}
p1.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, v in (("L", L), ("R", R)):
    pd.concat(v["predicciones"].values(), ignore_index=True).to_csv(
        p1.OUTPUT_DIR / f"predicciones_{name}.csv.gz", index=False)
    v["alfas"].to_csv(p1.OUTPUT_DIR / f"alfas_{name}.csv", index=False)

## 3 · Veredicto preregistrado (2022 y 2023)

In [ ]:
dec = p1.decision_1a(L, R)
dec["veredicto"]

In [ ]:
dec["antelacion"]

In [ ]:
dec["metricas_L_R"].round(3)

In [ ]:
print("Se adopta L como base de las fases siguientes:" if dec["adopta_L"] else
      "L compensa o pierde en algún horizonte: se detiene el estudio y se consulta a Rosa.", dec["adopta_L"])

## 4 · Lecturas que no deciden

In [ ]:
rd = p1.readings_1a(L, R, V1, inp["serie"], dec["medianas_L"])
rd["metricas_por_temporada_L"][["horizonte", "bloque", "positivos", "f1_050", "f1_050_rango", "recall_050",
    "precision_050", "auprc", "brier", "brier_sin_calibrar", "razon_mae", "razon_rmse"]].round(3)

In [ ]:
rd["hu0007_4_L"].round(3)

In [ ]:
rd["calibracion_V1_frente_a_R"].round(3)

In [ ]:
rd["recall_por_tipo_L"].round(3)

In [ ]:
rd["antelacion_L_semilla_17"].round(3)

In [ ]:
rd["conteo_por_fase_L"][["horizonte", "bloque", "fase", "n", "razon_mae", "razon_rmse", "mae_modelo",
    "rmse_modelo"]].round(3)

In [ ]:
rd["calibradores_L_semilla_17"]

## 5 · Salidas: métricas y MLflow

In [ ]:
content = {"dispositivo": device, "guardas": g, "adopta_L": dec["adopta_L"],
           **{k: diag.to_json(v) for k, v in dec.items() if isinstance(v, pd.DataFrame)},
           **{k: diag.to_json(v) for k, v in rd.items()}}
for section, value in content.items():
    diag.save_metrics_section(METRICS, section, value)
cell = lambda t, b, h, c: float(t[t.bloque.eq(b) & t.horizonte.eq(h)].iloc[0][c])
key = {f"{m}_{v}.{b[-4:]}.h{h}": cell(t, b, h, m)
       for v, t in (("L", dec["medianas_L"]), ("R", dec["medianas_R"]))
       for b in ("temporada_2022", "temporada_2023") for h in (2, 3, 4) for m in ("f1_050", "razon_mae")}
tracking.log_study(
    "mejora_modelo_fase1a_historia", pd.DataFrame(columns=["numero", "estado"]),
    parent={"params": {"fase": "1a", "preregistro": "6a29a8e", "enmienda_G1": "81e72a4",
                       "veredicto": "; ".join(f"h{r.horizonte}: {r.veredicto}" for r in dec["veredicto"].itertuples()),
                       "adopta_L": str(dec["adopta_L"]), **{f"xgb.{k}": v for k, v in device.items()}},
            "metricas": key},
    artifacts=[METRICS], tables={"veredicto": dec["veredicto"], "antelacion": dec["antelacion"]},
    description="Fase 1a del estudio 06: V1 entrenada desde 2005 frente a desde 2017.")